In [15]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [16]:
df = pd.read_csv("test (1).csv", encoding="latin1")
df.head()
                 

,textID,text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,f87dea47db,Last session of the day http://twitpic.com/67ezh,neutral,morning,0-20,Afghanistan,38928346.0,652860.0,60.0
1,96d74cb729,Shanghai is also really exciting (precisely -...,positive,noon,21-30,Albania,2877797.0,27400.0,105.0
2,eee518ae67,"Recession hit Veronique Branquinho, she has to...",negative,night,31-45,Algeria,43851044.0,2381740.0,18.0
3,01082688c6,happy bday!,positive,morning,46-60,Andorra,77265.0,470.0,164.0
4,33987a8ee5,http://twitpic.com/4w75p - I like it!!,positive,noon,60-70,Angola,32866272.0,1246700.0,26.0


In [17]:
df.isnull().sum()

textID              1281
text                1281
sentiment           1281
Time of Tweet       1281
Age of User         1281
Country             1281
Population -2020    1281
Land Area (Km²)     1281
Density (P/Km²)     1281
dtype: int64

In [18]:
print(df["text"].isnull().sum())

1281


In [19]:
df["text"] = df["text"].fillna("")

In [20]:
df["sentiment"] = df["sentiment"].fillna(df["sentiment"].mode()[0])

In [21]:
nltk.download("stopwords")
stop_word=set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [23]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\s+|www\s+","",text)
    text = re.sub(r"[^a-z\s]", "", text)        
    text = " ".join(word for word in text.split() if word not in stop_word)
    return text
df["clean_text"] = df["text"].apply(clean_text)
df[["text", "clean_text"]].head()

,text,clean_text
0,Last session of the day http://twitpic.com/67ezh,last session day httptwitpiccomezh
1,Shanghai is also really exciting (precisely -...,shanghai also really exciting precisely skyscr...
2,"Recession hit Veronique Branquinho, she has to...",recession hit veronique branquinho quit compan...
3,happy bday!,happy bday
4,http://twitpic.com/4w75p - I like it!!,httptwitpiccomwp like


In [24]:
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df["clean_text"])
y = df["sentiment"]

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [26]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


LogisticRegression(max_iter=1000)

In [27]:
y_pred = model.predict(X_test)


In [28]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.6863966770508827

Classification Report:

              precision    recall  f1-score   support

    negative       0.74      0.32      0.45       200
     neutral       0.66      0.93      0.77       542
    positive       0.81      0.42      0.55       221

    accuracy                           0.69       963
   macro avg       0.74      0.56      0.59       963
weighted avg       0.71      0.69      0.65       963


Confusion Matrix:

[[ 64 131   5]
 [ 20 505  17]
 [  3 126  92]]
